Prueba

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo2\\datosNarmax\\1pasos_mlp_consumption.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,temp,zone1,zone2,zone3,hour,e
date,,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858,NaN
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858,NaN
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858,NaN
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858,NaN
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858,NaN


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [6]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [7]:
futuros = 1
pasados  = 12

In [8]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 1])


In [9]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52404, 12, 6)
Dimensiones de Y: (52404, 1)


In [10]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [11]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (52404, 72)


Se dividen nuevamente los conjuntos de datos

In [12]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36682, 72)
Las dimensiones de testX son:  (10533, 72)
Las dimensiones de valX son:  (5189, 72)


In [13]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36682, 1)
Las dimensiones de testY son:  (10533, 1)
Las dimensiones de valY son:  (5189, 1)


Se crean métricas para medir desempeño

In [14]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

Versión Final


In [15]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=128,
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])

    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [16]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

58/58 - 4s - 73ms/step - ia: 0.2686 - loss: 1.0800 - mae: 0.8502 - rmse: 1.0385 - smape: 1.4587 - val_ia: 0.3356 - val_loss: 1.0947 - val_mae: 0.8874 - val_rmse: 1.0349 - val_smape: 1.8170

Epoch 2/128                                           

58/58 - 0s - 4ms/step - ia: 0.2629 - loss: 1.0220 - mae: 0.8250 - rmse: 1.0097 - smape: 1.4705 - val_ia: 0.3343 - val_loss: 1.0380 - val_mae: 0.8629 - val_rmse: 1.0079 - val_smape: 1.8500

Epoch 3/128                                           

58/58 - 0s - 6ms/step - ia: 0.2693 - loss: 0.9936 - mae: 0.8148 - rmse: 0.9953 - smape: 1.4657 - val_ia: 0.3334 - val_loss: 0.9962 - val_mae: 0.8444 - val_rmse: 0.9875 - val_smape: 1.8791

Epoch 4/128                                           

58/58 - 0s - 5ms/step - ia: 0.2632 - loss: 0.9690 - mae: 0.8021 - rmse: 0.9830 - smape: 1.4749 - val_ia: 0.3351 - val_loss: 0.9804 - val_mae: 0.8373 - val_rmse: 0.9796 - val_smape: 1.8815

Epoch 5/128        

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                     

461/461 - 5s - 11ms/step - ia: 0.9417 - loss: 0.0172 - mae: 0.0805 - rmse: 0.1025 - smape: 0.2147 - val_ia: 0.7055 - val_loss: 0.0117 - val_mae: 0.0842 - val_rmse: 0.0951 - val_smape: 0.2367

Epoch 2/128                                                                     

461/461 - 2s - 3ms/step - ia: 0.9614 - loss: 0.0050 - mae: 0.0537 - rmse: 0.0676 - smape: 0.1668 - val_ia: 0.7439 - val_loss: 0.0068 - val_mae: 0.0640 - val_rmse: 0.0753 - val_smape: 0.1403

Epoch 3/128                                                                     

461/461 - 2s - 4ms/step - ia: 0.9603 - loss: 0.0053 - mae: 0.0553 - rmse: 0.0693 - smape: 0.1649 - val_ia: 0.8247 - val_loss: 0.0034 - val_mae: 0.0402 - val_rmse: 0.0526 - val_smape: 0.1200

Epoch 4/128                                                                     

461/461 - 1s - 3ms/step - ia: 0.9641 - loss: 0.0044 - mae: 0.0500 - rmse: 0.0635 - smape: 0.1545 -

...

461/461 - 9s - 20ms/step - ia: 0.5023 - loss: 0.7107 - mae: 0.6648 - rmse: 0.8211 - smape: 1.1264 - val_ia: 0.2795 - val_loss: 0.3100 - val_mae: 0.4808 - val_rmse: 0.5070 - val_smape: 0.9022

Epoch 2/128                                                                        

461/461 - 1s - 2ms/step - ia: 0.6320 - loss: 0.4347 - mae: 0.5222 - rmse: 0.6475 - smape: 0.9123 - val_ia: 0.3367 - val_loss: 0.1711 - val_mae: 0.3557 - val_rmse: 0.3775 - val_smape: 0.7526

Epoch 3/128                                                                        

461/461 - 1s - 3ms/step - ia: 0.6837 - loss: 0.3123 - mae: 0.4436 - rmse: 0.5493 - smape: 0.8190 - val_ia: 0.3649 - val_loss: 0.1268 - val_mae: 0.3111 - val_rmse: 0.3298 - val_smape: 0.6909

Epoch 4/128                                                                        

461/461 - 1s - 2ms/step - ia: 0.7255 - loss: 0.2324 - mae: 0.3803 - rmse: 0.4741 - smape: 0.7453 - val_ia: 0.3985 - val_loss: 0.0945 - val_mae: 0.2682 - val_rmse: 0.2862 -

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



922/922 - 4s - 5ms/step - ia: 0.3013 - loss: 2.2524 - mae: 1.0595 - rmse: 1.3905 - smape: 1.5329 - val_ia: 0.1276 - val_loss: 0.6873 - val_mae: 0.7188 - val_rmse: 0.7368 - val_smape: 1.3957

Epoch 2/128                                                                         

922/922 - 2s - 2ms/step - ia: 0.3044 - loss: 2.1245 - mae: 1.0375 - rmse: 1.3549 - smape: 1.5313 - val_ia: 0.1265 - val_loss: 0.6925 - val_mae: 0.7218 - val_rmse: 0.7396 - val_smape: 1.4168

Epoch 3/128                                                                         

922/922 - 2s - 2ms/step - ia: 0.2931 - loss: 2.0405 - mae: 1.0306 - rmse: 1.3315 - smape: 1.5497 - val_ia: 0.1250 - val_loss: 0.6981 - val_mae: 0.7250 - val_rmse: 0.7424 - val_smape: 1.4380

Epoch 4/128                                                                         

922/922 - 3s - 3ms/step - ia: 0.3025 - loss: 1.9635 - mae: 1.0075 - rmse: 1.3018 - smape: 1.5432 - val_ia: 0.1235 - val_loss: 0.7045 - val_mae: 0.7284 - val_rmse: 0.7455

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



58/58 - 2s - 35ms/step - ia: 0.6006 - loss: 0.5129 - mae: 0.5643 - rmse: 0.6942 - smape: 0.9781 - val_ia: 0.7494 - val_loss: 0.1824 - val_mae: 0.3486 - val_rmse: 0.4204 - val_smape: 0.7511

Epoch 2/128                                                                         

58/58 - 0s - 3ms/step - ia: 0.7293 - loss: 0.2479 - mae: 0.3957 - rmse: 0.4967 - smape: 0.7658 - val_ia: 0.8048 - val_loss: 0.1029 - val_mae: 0.2682 - val_rmse: 0.3142 - val_smape: 0.6569

Epoch 3/128                                                                         

58/58 - 0s - 3ms/step - ia: 0.7595 - loss: 0.2002 - mae: 0.3546 - rmse: 0.4456 - smape: 0.7099 - val_ia: 0.8428 - val_loss: 0.0671 - val_mae: 0.2166 - val_rmse: 0.2542 - val_smape: 0.5687

Epoch 4/128                                                                         

58/58 - 0s - 5ms/step - ia: 0.7801 - loss: 0.1670 - mae: 0.3253 - rmse: 0.4080 - smape: 0.6644 - val_ia: 0.8740 - val_loss: 0.0438 - val_mae: 0.1713 - val_rmse: 0.2063 - val_

In [17]:
print(best)

{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}


In [18]:
#{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}